# Null calibration: is any atom an artifact of the estimator?

## The null

Permute **S within strata of V**. That makes `S ⟂ Y | V` hold by construction, so
the true unique-to-structure and the true synergy are both exactly zero, while
`p(V,S)`, `p(V,Y)`, the level counts and the dimensionality are all preserved.
Anything the estimator reports under that null is bias, measured on the real
data's geometry rather than on a synthetic toy.

This is a sharper test than the permutation gate in `01_significance_gate`, which
shuffles Y against fixed (V,S) cells. That null destroys the V–Y association too,
so it cannot detect a bias that only appears while that association is intact.

This notebook runs the **same five seeds as the headline** and reports the floor
per seed and averaged, so `debiased = observed − floor` is a like-for-like
subtraction. Seed 0 is kept separately so the original file's numbers stay
traceable.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
from multiprocessing import get_context
from pathlib import Path

import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import pid_broja as pb
import quantize as qz

pd.set_option("display.width", 200)

K = 13
SEEDS = [0, 1, 2, 3, 4]          # the headline's seeds -- the point of this rebuild
N_PERM = 200                     # per seed, so 1,000 draws per cell in total
MEASURES = ("broja", "ccs")
ATOMS = ("red", "u_val", "u_str", "syn")
N_JOBS = 16

MG_MIMIC = 50960
PANEL12_EICU = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
                "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}
TARGETS = ["mortality", "gender_female", "age_ge65", "race_binary"]

cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG_MIMIC})
cc_e = eicu.complete_case(labname_scope=PANEL12_EICU)
DBS = [("MIMIC-IV", mimic, cc_m), ("eICU-CRD", eicu, cc_e)]
for db, _, cc in DBS:
    print(f"{db:9s} n={cc.meta['n']:7,d}  V={cc.Xv.shape[1]}d S={cc.Xs.shape[1]}d")
print(f"\nk={K}, seeds={SEEDS}, {N_PERM} permutations per seed "
      f"-> {N_PERM*len(SEEDS):,} draws per cell")

MIMIC-IV  n= 64,623  V=24d S=24d
eICU-CRD  n=120,266  V=24d S=24d

k=13, seeds=[0, 1, 2, 3, 4], 200 permutations per seed -> 1,000 draws per cell


## 1. The conditional-independence null

Permuting S only *within* a V stratum leaves the (V,S) joint counts and the (V,Y)
joint counts untouched, so the only thing destroyed is any S–Y association that
is not already carried by V.

In [2]:
def _strata(v):
    """Row indices grouped by V level. Computed once per grid, not per draw."""
    order = np.argsort(v, kind="stable")
    return np.split(order, np.flatnonzero(np.diff(v[order])) + 1)


def _null_batch(args):
    """One worker task: `n` conditional-independence draws on a fixed grid.

    Returns atoms as % of joint for both measures. A draw whose solver cascade
    fails comes back as NaN from `decompose` and is filtered at the aggregation
    step rather than here, so the loss stays visible."""
    v, s, y, groups, seed, n = args
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        s2 = s.copy()
        for g in groups:
            s2[g] = s[rng.permutation(g)]
        row = {}
        for meas in MEASURES:
            d = pb.decompose(v, s2, y, measure=meas)
            for a in ATOMS:
                row[f"{meas}_{a}"] = d["atoms_pct_of_joint"][a]
        out.append(row)
    return out


# Pool created AFTER the worker is defined: the fork start method snapshots the
# parent's memory, so a function defined later would not exist in the children.
POOL = get_context("fork").Pool(N_JOBS)
print(f"pool of {N_JOBS} workers ready")

pool of 16 workers ready


## 2. Run: 8 cells x 5 seeds x 200 permutations

In [3]:
# SHARED CODEBOOK (2026-09-01): one k-means fit per (database, seed) on the FULL
# cohort, masked to each target's rows, so every outcome is described in the same
# vocabulary. Must match notebooks/02 -- a floor or a variance split measured
# under a different discretisation is not comparable to the headline it refers to.
CODEBOOK, DEMO = {}, {}
for _db, _mod, _cc in DBS:
    DEMO[_db] = _mod.demographic_targets()[0]
    for _sd in SEEDS:
        _v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(_cc.Xv), K, seed=_sd)
        _s, _ = qz.kmeans_levels(
            qz.standardize_for_kmeans(_cc.Xs, sentinel=_mod.TIME_SENTINEL), K, seed=_sd)
        CODEBOOK[(_db, _sd)] = (_v, _s)
print(f"codebooks: {len(CODEBOOK)} fit on the full cohorts")

BATCH = 25
rows, perseed = [], []
t_all = time.time()

for db, mod, cc in DBS:
    demo = DEMO[db]
    for target in TARGETS:
        ch = cc if target == "mortality" else mod.retarget(cc, demo[target], target)
        keep = (None if target == "mortality"
                else demo[target].reindex(pd.Index(cc.groups)).notna().to_numpy())
        t0 = time.time()
        obs_by_seed, null_by_seed = [], []

        for sd in SEEDS:
            v, s = CODEBOOK[(db, sd)]
            if keep is not None: v, s = v[keep], s[keep]
            groups = _strata(v)

            obs = {}
            for meas in MEASURES:
                d = pb.decompose(v, s, ch.y, measure=meas)
                for a in ATOMS:
                    obs[f"{meas}_{a}"] = d["atoms_pct_of_joint"][a]
            obs_by_seed.append(obs)

            tasks = [(v, s, ch.y, groups, sd * 10_000 + i, BATCH)
                     for i in range(N_PERM // BATCH)]
            draws = pd.DataFrame([r for chunk in POOL.map(_null_batch, tasks) for r in chunk])
            null_by_seed.append(draws)
            perseed.append({"db": db, "target": target, "seed": sd,
                            **{f"obs_{k}": v_ for k, v_ in obs.items()},
                            **{f"nullmean_{c}": float(draws[c].mean()) for c in draws.columns}})

        obs_df = pd.DataFrame(obs_by_seed)
        allnull = pd.concat(null_by_seed, ignore_index=True)
        for meas in MEASURES:
            rec = {"db": db, "target": target, "measure": meas,
                   "k": K, "seeds": SEEDS, "n_perm_per_seed": N_PERM}
            for a in ATOMS:
                col = f"{meas}_{a}"
                nul = allnull[col].to_numpy()
                fin = nul[np.isfinite(nul)]
                o5 = float(obs_df[col].mean())
                o0 = float(obs_df[col].iloc[0])
                rec[f"obs_{a}"] = o5                       # 5-seed mean: headline-coherent
                rec[f"obs_{a}_seed0"] = o0                 # traceable to the lost original
                rec[f"obs_{a}_sd"] = float(obs_df[col].std())
                rec[f"null_{a}_mean"] = float(fin.mean())
                rec[f"null_{a}_p95"] = float(np.quantile(fin, 0.95))
                rec[f"null_{a}_max"] = float(fin.max())
                rec[f"debiased_{a}"] = o5 - float(fin.mean())
                rec["n_ok"] = int(fin.size)
                rec["n_dropped"] = int(nul.size - fin.size)
            rows.append(rec)

        print(f"{db:9s} {target:14s} {time.time()-t0:6.1f}s   "
              f"broja u_str floor={rows[-2]['null_u_str_mean']:+.4f}  "
              f"syn floor={rows[-2]['null_syn_mean']:5.2f}", flush=True)

POOL.close(); POOL.join()
cal = pd.DataFrame(rows)
print(f"\ntotal {time.time()-t_all:.0f}s")

codebooks: 10 fit on the full cohorts


MIMIC-IV  mortality        94.2s   broja u_str floor=+0.0000  syn floor= 4.56


MIMIC-IV  gender_female    94.3s   broja u_str floor=+0.0000  syn floor= 5.97


MIMIC-IV  age_ge65         94.7s   broja u_str floor=+0.0000  syn floor= 6.84


MIMIC-IV  race_binary      94.7s   broja u_str floor=+0.0000  syn floor=19.62


eICU-CRD  mortality        95.6s   broja u_str floor=-0.0000  syn floor= 4.32


eICU-CRD  gender_female    95.9s   broja u_str floor=-0.0000  syn floor= 2.82


eICU-CRD  age_ge65         95.7s   broja u_str floor=-0.0000  syn floor= 2.55


eICU-CRD  race_binary      95.0s   broja u_str floor=-0.0000  syn floor=10.10



total 760s


## 3. What the calibration says

In [4]:
b = cal[cal.measure == "broja"]
c = cal[cal.measure == "ccs"]
print("BROJA — unique-to-structure floor (true value is 0 by construction)\n")
print(b[["db","target","obs_u_str","null_u_str_mean","null_u_str_max","debiased_u_str"]]
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  max |BROJA u_str bias| across all cells: {b.null_u_str_mean.abs().max():.5f}"
      f"   (max single draw {b.null_u_str_max.abs().max():.5f})")
print(f"  CCS  u_str null means span {c.null_u_str_mean.min():.2f} to {c.null_u_str_mean.max():.2f}"
      f"  -- a large, cell-varying offset, not correctable by subtracting a mean")

print("\n\nSYNERGY — floor and net value\n")
print(cal[["db","target","measure","obs_syn","null_syn_mean","debiased_syn"]]
      .to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print(f"\n  synergy floor spans {cal.null_syn_mean.min():.1f} to {cal.null_syn_mean.max():.1f} points")
print(f"  net of floor, synergy spans {cal.debiased_syn.min():.1f} to {cal.debiased_syn.max():.1f}")

race = cal[cal.target == "race_binary"] if "race_binary" in set(cal.target) else cal.iloc[:0]
mr = race[(race.db == "MIMIC-IV")]
if len(mr) == 2:
    bb = mr[mr.measure=="broja"].iloc[0]; cc_ = mr[mr.measure=="ccs"].iloc[0]
    print(f"\n  MIMIC race synergy, raw:      BROJA {bb.obs_syn:.2f}  CCS {cc_.obs_syn:.2f}"
          f"   (differ by {abs(bb.obs_syn-cc_.obs_syn):.2f})")
    print(f"  MIMIC race synergy, debiased: BROJA {bb.debiased_syn:.2f}  CCS {cc_.debiased_syn:.2f}"
          f"   (differ by {abs(bb.debiased_syn-cc_.debiased_syn):.2f})")

print(f"\n  seed sensitivity, BROJA u_str: mean |5-seed - seed0| = "
      f"{(b.obs_u_str - b.obs_u_str_seed0).abs().mean():.3f} points")
print(f"  seed sensitivity, BROJA syn  : mean |5-seed - seed0| = "
      f"{(b.obs_syn - b.obs_syn_seed0).abs().mean():.3f} points")

BROJA — unique-to-structure floor (true value is 0 by construction)

      db        target  obs_u_str  null_u_str_mean  null_u_str_max  debiased_u_str
MIMIC-IV     mortality     4.1613           0.0000          0.0001          4.1613
MIMIC-IV gender_female     0.0890           0.0000          0.0276          0.0890
MIMIC-IV      age_ge65    -0.0000           0.0000          0.0005         -0.0000
MIMIC-IV   race_binary    12.5204           0.0000          0.0051         12.5204
eICU-CRD     mortality     0.1413          -0.0000         -0.0000          0.1413
eICU-CRD gender_female    -0.0000          -0.0000         -0.0000          0.0000
eICU-CRD      age_ge65    -0.0000          -0.0000         -0.0000          0.0000
eICU-CRD   race_binary    -0.0000          -0.0000         -0.0000          0.0000

  max |BROJA u_str bias| across all cells: 0.00003   (max single draw 0.02757)
  CCS  u_str null means span -9.97 to -3.78  -- a large, cell-varying offset, not correctable by subtrac

## 4. Save

In [5]:
# Numbers only. The findings this file supports -- BROJA's u_str being unbiased,
# CCS's being unusable, synergy carrying a floor -- are stated in the manuscript,
# where they can be argued. What is saved here is what they are read off.
#
# `method`, `provenance` and `aggregation` stay because they are not
# interpretation: without them the numbers cannot be read at all. `summary` is
# the quantitative content of the old prose headline, kept as numbers so it
# cannot drift from `cells`.
payload = {
    "method": (f"Conditional-independence null: permute S within strata of V so S _||_ Y | V "
               f"holds by construction and the TRUE u_str and syn are exactly 0, while p(V,S), "
               f"p(V,Y), level counts and dimensionality are preserved. Anything reported under "
               f"this null is bias, measured on real data geometry. {N_PERM} permutations per "
               f"seed x {len(SEEDS)} seeds = {N_PERM*len(SEEDS)} draws per cell, k={K}, matched "
               f"12-analyte panel, 16 cells (2 databases x 4 outcomes x 2 measures). Permutes "
               f"discretised levels, so it isolates the PID estimator and plug-in table and does "
               f"not capture the k-means contribution."),
    "provenance": ("Rebuilt by notebooks/05_null_calibration.ipynb. The original was generated "
                   "ad hoc on 2026-08-24 at SEED 0 ONLY and its script was not kept; observed "
                   "atoms here are the 5-seed mean matching "
                   "results/headline_matched_panel12.json, with obs_*_seed0 retained so the "
                   "original numbers remain traceable."),
    "aggregation": ("obs_* are the mean over the headline's 5 k-means seeds; obs_*_seed0 is the "
                    "single-seed value the lost original reported. debiased_* = obs_* (5-seed) "
                    "minus null_*_mean, so the subtraction is like-for-like. The floor applies "
                    "to u_str and syn only -- the null forces those two to zero by construction "
                    "but leaves red and u_val carrying the null's own composition."),
    "summary": {
        "broja_u_str_max_abs_null_mean": float(b.null_u_str_mean.abs().max()),
        "broja_u_str_max_abs_null_single_draw": float(b.null_u_str_max.abs().max()),
        "ccs_u_str_null_mean_min": float(c.null_u_str_mean.min()),
        "ccs_u_str_null_mean_max": float(c.null_u_str_mean.max()),
        "syn_null_mean_min": float(cal.null_syn_mean.min()),
        "syn_null_mean_max": float(cal.null_syn_mean.max()),
        "syn_debiased_min": float(cal.debiased_syn.min()),
        "syn_debiased_max": float(cal.debiased_syn.max()),
        "seed_sensitivity_u_str_mean_abs_5seed_minus_seed0":
            float((b.obs_u_str - b.obs_u_str_seed0).abs().mean()),
        "seed_sensitivity_syn_mean_abs_5seed_minus_seed0":
            float((b.obs_syn - b.obs_syn_seed0).abs().mean()),
    },
    "cells": rows,
    "per_seed": perseed,
}
out = Path("../results/pid_null_calibration.json")
out.write_text(json.dumps(payload, indent=2, default=float))
print(f"wrote {out}  ({len(rows)} cells, {len(perseed)} per-seed records, numbers only)")


wrote ../results/pid_null_calibration.json  (16 cells, 40 per-seed records, numbers only)
